In [ ]:
import os
import itertools
import warnings
import logging
import random

import torch
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
import numpy as np

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.molmim.infer import MolMIMInference
from nemo.collections.common.tokenizers.regex_tokenizer import RegExTokenizer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

In [ ]:
train_df = pd.read_csv("/workspace/bionemo/data/train_substrate.csv")
test_df = pd.read_csv("/workspace/bionemo/data/test_substrate.csv")

train_df.shape, test_df.shape

In [ ]:
max_token_length = 126
# Note: the maximum token length generated from the smiles string should be 2 less than the max_seq_length specified in the model config.
# This is to account for the extra tokens <BOS> and <EOS>

def vocab_compliance_check(smiles: str, tokenizer: RegExTokenizer, max_token_length: int) -> bool:
    """Checks if the SMILES string only contains vocabulary in the tokenizer's vocabulary
    and if the token length is less than or equal to `max_token_length"""
    tokens = tokenizer.text_to_tokens(smiles)
    vocab_allowed = tokenizer.vocab.keys()
    return set(tokens).issubset(set(vocab_allowed)) and len(tokens) <= max_token_length

model_name = "molmim"
print(f"Filtering out molecules which are not present in the {model_name} tokenizer vocabulary or with max token length greater than {max_token_length}...")
tokenizer_path = bionemo_home + "/tokenizers/molecule/{model_name}/vocab/{model_name}.{extension}"
tokenizer = RegExTokenizer().load_tokenizer(regex_file=tokenizer_path.format(model_name=model_name, extension="model"), vocab_file=tokenizer_path.format(model_name=model_name, extension="vocab"))

train_df["vocab_compliant"] = train_df["canon_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
test_df["vocab_compliant"] = test_df["canon_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
train_df = train_df.loc[train_df['vocab_compliant']]
test_df = test_df.loc[test_df['vocab_compliant']]

print(f"{len(train_df)} molecules in train set after filtering.")
print(f"{len(test_df)} molecules in test set after filtering.")

In [ ]:
# os.makedirs("/workspace/bionemo/data/processed/train", exist_ok=True)
# os.makedirs("/workspace/bionemo/data/processed/val", exist_ok=True)

# train_df.to_csv("/workspace/bionemo/data/processed/train/train.csv", index=False)
# test_df.to_csv("/workspace/bionemo/data/processed/val/test.csv", index=False)

# fake test df
pd.DataFrame({"canon_smiles": ["C1=CC=CC=C1", "C1=CC=CC=C1O"]}).to_csv("/workspace/bionemo/data/processed/finetuning/test/test.csv", index=False)

In [ ]:
import subprocess

# --- Customizable Parameters ---
# Model and Data Paths
model_path = f"{bionemo_home}/models/molecule/molmim/molmim_70m_24_3.nemo"
task = "finetuning"  # Specify your task name
index_mapping_dir = "data/data_index/"
data_col = 0
config_name: str = "pretrain_small_canonicalized_logv"

# Dataset Configuration
train_set = "train"
val_set = "val"
test_set = "test"

# Training Parameters
do_training = True
devices = 1
accelerator = 'cpu'
max_steps = 6
val_check_interval = 2
global_batch_size = "null"  # Use "null" as a string to be correctly interpreted in the command

# Experiment Management
create_wandb_logger = True
resume_if_exists = False

# --- Execution ---
# Clean up previous data index
if os.path.exists(index_mapping_dir):
    subprocess.run(["rm", "-rf", index_mapping_dir], check=True)


# Construct the pre-training command using an f-string
command = (
    f"python {bionemo_home}/examples/molecule/molmim/pretrain.py "
    f"--config-path=/workspace/bionemo/examples/molecule/molmim/conf "
    f"--config-name={config_name} "
    f"restore_from_path={model_path} "  # <-- KEY: Load the vanilla model
    f"do_training={do_training} "
    f"++model.data.dataset_path=data/processed/{task}/ "
    f"++model.data.dataset.train={train_set} "
    f"++model.data.dataset.val={val_set} "
    f"++model.data.dataset.test={test_set} "
    f"++model.data.index_mapping_dir={index_mapping_dir} "
    f"++model.data.data_impl_kwargs.csv_mmap.data_col={data_col} "
    f"++model.dwnstr_task_validation.enabled=False "
    f"++model.global_batch_size={global_batch_size} "
    f"++trainer.devices={devices} "
    f"++trainer.accelerator='{accelerator}' "
    f"++trainer.max_steps={max_steps} "
    f"++trainer.val_check_interval={val_check_interval} "
    f"++exp_manager.create_wandb_logger={create_wandb_logger} "
    f"++exp_manager.resume_if_exists={resume_if_exists} "
    f"++trainer.precision=bf16-mixed "
    # 2. Force the DataLoader to keep data on the CPU
    "++model.data.pin_memory=False "
    # 3. Switch from the GPU-only 'fused_adam' to a standard CPU-compatible optimizer
    "++model.optim.name=adamw "
    # 4. Disable GPU-specific CUDA kernel fusions
    "++model.megatron_amp_O2=False "
    "++model.bias_gelu_fusion=False "
)

command

In [ ]:
import csv
from rdkit import Chem

# --- Configure this to match your validation file ---
validation_file_path = "data/processed/finetuning/test/test.csv" # Add .csv if needed
smiles_column_index = 0 # The column containing SMILES (0-indexed)

print(f"Checking for invalid SMILES in: {validation_file_path}")
invalid_smiles_count = 0

with open(validation_file_path, 'r', newline='') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if not row: # Skip empty rows
            print(f"Line {i+1}: Empty row found.")
            invalid_smiles_count += 1
            continue
        try:
            smiles = row[smiles_column_index]
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                # MolFromSmiles returns None for invalid SMILES without raising an error
                print(f"Line {i+1}: Invalid SMILES string found: '{smiles}'")
                invalid_smiles_count += 1
        except IndexError:
            print(f"Line {i+1}: Row is too short, cannot find column {smiles_column_index}. Content: {row}")
            invalid_smiles_count += 1
        except Exception as e:
            # Catch any other unexpected errors
            print(f"Line {i+1}: An unexpected error occurred: {e}")
            invalid_smiles_count += 1


if invalid_smiles_count == 0:
    print("\nNo invalid SMILES found. The file appears to be correctly formatted.")
else:
    print(f"\nFound a total of {invalid_smiles_count} problematic lines.")